# Análise do IDH por Unidade Federativa

Estudo da variação do indicador entre 1991 e 2024 com Pandas e Matplotlib.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

arquivo = Path("Tabela4.csv")
df = pd.read_csv(arquivo, sep=";", decimal=",", encoding="latin1", header=1)
df = df.dropna(axis=1, how="all")
df.columns = df.columns.astype(str).str.strip()
df.head()

In [ ]:
anos = [c for c in df.columns if c.isnumeric()]
df[anos] = df[anos].apply(pd.to_numeric, errors="coerce")
print("Anos encontrados:", anos)


In [ ]:
ranking = df[["Sigla", "Estado", "2024"]].sort_values("2024", ascending=False)
ranking.reset_index(drop=True)

In [ ]:
df["variacao_idh"] = df["2024"] - df["1991"]
resultado = df.loc[df["variacao_idh"].idxmax()]
print(f"Maior avanço: {resultado['Estado']} ({resultado['Sigla']})")
print(f"1991 = {resultado['1991']:.3f}")
print(f"2024 = {resultado['2024']:.3f}")
print(f"Variação = {resultado['variacao_idh']:.3f}")

In [ ]:
quedas = df.loc[df["variacao_idh"] < 0, ["Sigla", "Estado", "1991", "2024", "variacao_idh"]]
if quedas.empty:
    print("Não houve redução do IDH no período analisado.")
else:
    display(quedas)

In [ ]:
info = [c for c in df.columns if c not in anos]
historico = df.melt(
    id_vars=info,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH"
)
historico["Ano"] = historico["Ano"].astype(int)
historico["IDH"] = pd.to_numeric(historico["IDH"], errors="coerce")
historico.head()

In [ ]:
mg = historico.loc[historico["Sigla"].eq("MG")].sort_values("Ano")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2)
ax.set_title("Evolução do IDH de Minas Gerais")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
for uf, grupo in historico.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=uf)
ax.set_title("Evolução do IDH por Unidade Federativa")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()

## Conclusões

A maior variação positiva entre 1991 e 2024 ocorre em Tocantins, que passou de 0,369 para 0,797. A comparação entre os dois anos não identifica nenhuma unidade federativa com redução do IDH. O segundo gráfico permite observar a trajetória do indicador ao longo do período para todas as UFs.